# Lucia 0.8B v1.2: the Apache-2.0 and non-commercial lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/lucia_v12_08b.ipynb)

The v1.2 fine-tune of [Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B) on the rows of the public dataset [thought-symbolic-image-recaptioned](https://huggingface.co/datasets/AbstractPhil/thought-symbolic-image-recaptioned) (release 5), held near the base model's behavior while it learns them (the general-row KL at weight 1, on general requests the base model answered itself), in one pass over the rows. The Apache-2.0 line trains on the dataset's Apache-2.0 part; the non-commercial line (-nc) on every row of the release; the held-out pictures are the same for both, so their reads compare.

The queue runs, in order: v11a_08b; v11a_08b_by; v11txt_08b; the smoke S_v12_smoke_gc1_08b; the training V12_A1_gkl1_1ep_08b_by and its nine reads; XT_base_08b_noadapter; XT_base_08b_deep; XTA_base_08b; XCA2_base_08b; XC2_base_08b; K0_lucia1_08b; XT_v1_08b_s1_deep; XT_v1_08b_s1; the smoke S_v12_smoke_gc1_08b_nc; the training V12NC_A1_gkl1_1ep_08b and its nine reads. A smoke that runs slower or holds more memory than its limits fails, and its training does not start. The apache line about 5.9 h of training at 2.1 s a step (estimated); the nc line about 11.5 h of training at 1.9 s a step (estimated).

The training runs with resume on: its whole state saves to the project repo every 2 hours and once more after the runtime's 23rd hour. If the runtime ends mid-run, open this notebook on a fresh runtime and run it again: the lane continues the training from its saved state. Its progress file (runs/progress/<run>.json in the project repo) says where it stands: `python -m qwen_symbolic_thinking.tools.progress <project repo>`.

Runtime: a GPU (the RTX PRO 6000, 95 GB) and the `HF_TOKEN` secret with write access to the project repo `AbstractPhil/qwen-symbolic-thinking-image-training-private`, where every file of this lane lives (its training sets and inputs first: see the package's docs/registry_and_lanes.md). Run C1, restart the session (Runtime > Restart session), then run C2.


In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.68", "transformers>=5.0", "accelerate>=1.0", "peft>=0.18")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (training still runs, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

try:                # Colab's torchao 0.10 breaks PEFT when it builds a LoRA (PEFT wants 0.16 or newer, or none at all)
    import torchao
    _v = tuple(int(x) for x in torchao.__version__.split(".")[:2])
    if _v < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", torchao.__version__, "removed: PEFT cannot build a LoRA beside it")
    else:
        print("torchao", torchao.__version__, "stays")
except ImportError:
    print("no torchao")

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: work through the lane's queue (runs/queue_lucia_v12_08b.json in the project repo; new runs can be added while it runs)
import os
os.environ["QST_REPO"] = "AbstractPhil/qwen-symbolic-thinking-image-training-private"   # before the package is imported: every file of this lane lives in this repo
LANE = "lucia_v12_08b"

from qwen_symbolic_thinking import REPO, __version__
from qwen_symbolic_thinking.queue import run_queue

print("package", __version__, "| project repo", REPO, "| lane", LANE)
ran = run_queue(LANE, wait_hours=12)
print("ran:", ran)
